# PACE in five minutes

**Different Noise Levels, Different Needs: Discovering Diffusion Phases for Efficient Distillation**
Eitan Kosman, Gabriele Serussi, Chaim Baskin · [project page](https://gabrieleserussi.github.io/pace-diffusion/) · [code](https://github.com/GabrieleSerussi/pace-diffusion)

A diffusion denoiser relies on different parts of itself as the noise level falls. PACE measures this in a pretrained teacher, groups the noise levels into **phases** and distills **one smaller student per phase**. Each denoising call then runs a single student.

This notebook walks through the four steps on the paper's FFHQ-64 U-Net, from its released profile and with the repository's own scripts, and then shows how to run PACE on your own teacher. Everything runs on the free CPU runtime in about a minute.

In [ ]:
#@title Setup (about 20 seconds)
teacher = "FFHQ-64 U-Net"  #@param ["FFHQ-64 U-Net", "CIFAR-10 U-Net", "ImageNet-64 U-Net", "LSUN Bedroom-256 U-Net"]
my_profile = ""  #@param {type:"string"}
#@markdown Leave `my_profile` empty to use the released profile of `teacher`, or give the path of a `results.json` written by `scripts/evaluate_parameters_edm.py` for your own teacher (upload it with the file browser on the left).

import base64, io, json, os, re, subprocess, sys, tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

REPO = Path(os.environ.get("PACE_REPO", "/content/pace-diffusion")).resolve()
if not (REPO / "pace").is_dir():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/GabrieleSerussi/pace-diffusion", str(REPO)], check=True)
sys.path.insert(0, str(REPO))
from pace.capacity_allocation import compute_delta_p_eff_geomean_metric
from pace.jsonio import load_json

WORK = Path(tempfile.mkdtemp(prefix="pace_"))
COLORS = ["#3F6FB6", "#B7832A", "#3A9A8A", "#A8456B", "#6B5BA8", "#5B8C3A"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

# The four U-Net teachers of the paper: released profile, phase setting and sampler steps.
# The paper fixes three phases for FFHQ-64; the others use the default, automatic number of phases.
TEACHERS = {
    "FFHQ-64 U-Net": dict(stem="ffhq64_ddpmpp_pfi", phases=["--builtin_cost", "matrix_correlation", "--num_blocks", "3"], steps=40),
    "CIFAR-10 U-Net": dict(stem="cifar10_ddpmpp_random_same_norm", phases=[], steps=18),
    "ImageNet-64 U-Net": dict(stem="imagenet64_adm_random_same_norm", phases=[], steps=18),
    "LSUN Bedroom-256 U-Net": dict(stem="lsun256_adm_pfi_stratified", phases=[], steps=40),
}


def run(script, *args, echo=False):
    """Run one of the repository's scripts, exactly as you would from a shell."""
    proc = subprocess.Popen([sys.executable, str(REPO / "scripts" / script), *map(str, args)], cwd=REPO,
                            env=dict(os.environ, PYTHONPATH=str(REPO)), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if echo:
            print(line, end="")
    if proc.wait():
        raise RuntimeError(f"{script} failed:\n" + "".join(lines[-30:]))


if my_profile.strip():
    T, PROFILE, teacher = dict(stem="yours", phases=[], steps=40), Path(my_profile.strip()).expanduser(), "Your teacher"
else:
    T = TEACHERS[teacher]
    PROFILE = REPO / "artifacts" / "profiles" / f"{T['stem']}.json.gz"
profile = load_json(PROFILE)
print(f"{teacher}: profile of {len(profile['group_names']):,} channels at {len(profile['sigma_bin_labels'])} noise levels.")

## 1. The teacher uses different parts of itself at different noise levels

For every channel and every noise level, PACE gives the channel the activation of another image at the same noise level and records how much the denoising loss rises. Summed over each level of the U-Net, these loss increases show where the teacher does its work as the noise falls.

In [ ]:
#@title Where the teacher works, noise level by noise level
delta = np.clip(np.asarray(profile["delta_stack"], dtype=float), 0, None)   # loss rise: channels x noise levels
parts = {}
for i, name in enumerate(profile["group_names"]):
    m = re.match(r"model\.(enc|dec)\.(\d+)x\d+", name)
    if m:
        parts.setdefault(f"{'encoder' if m.group(1) == 'enc' else 'decoder'} {m.group(2)}x{m.group(2)}", []).append(i)

if parts:
    usage = np.stack([delta[idx].sum(0) for idx in parts.values()])
    usage /= usage.sum(0, keepdims=True)
    fig, ax = plt.subplots(figsize=(9, 3.3))
    im = ax.imshow(usage, aspect="auto", cmap="Blues", vmin=0)
    ax.set_yticks(range(len(parts)), list(parts))
    ax.set_xticks([0, usage.shape[1] - 1], ["high noise", "low noise"])
    ax.set_title("Share of the loss increase, by U-Net level")
    fig.colorbar(im, ax=ax, fraction=0.03)
    plt.show()
else:
    print("The channel names of this teacher do not record U-Net levels; step 2 shows its usage structure.")

## 2. Noise levels that use the teacher alike form a phase

PACE compares the usage profiles of every pair of noise levels. Blocks of high correlation along the diagonal are phases, and dynamic programming cuts the noise range into contiguous phases whose profiles are alike. This is one call to `scripts/optimize_timestep_grouping.py`.

In [ ]:
#@title Find the phases
run("optimize_timestep_grouping.py", "--matrix", PROFILE, *T["phases"], "--output", WORK / "phases.json")
bounds = json.loads((WORK / "phases.json").read_text())["boundaries"]
phases = list(zip(bounds[:-1], bounds[1:]))
S = np.asarray(profile["C_noise_levels"], dtype=float)   # Pearson correlation between usage profiles


def show_phases(S, phases, title):
    fig, ax = plt.subplots(figsize=(5, 4.4))
    im = ax.imshow(S, cmap="Blues", vmin=0, vmax=1)
    for k, (a, b) in enumerate(phases):
        ax.add_patch(plt.Rectangle((a - 0.5, a - 0.5), b - a, b - a, fill=False, lw=2.5, ec=COLORS[k % len(COLORS)]))
    ends = [0, len(S) - 1]
    ax.set_xticks(ends, ["high noise", "low noise"]); ax.set_yticks(ends, ["high noise", "low noise"])
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.show()
    print(f"{len(phases)} phases over {len(S)} noise levels:", ", ".join(f"[{a}, {b})" for a, b in phases))


show_phases(S, phases, "Similarity between noise levels, and the phases")

## 3. Each phase gets a share of the budget that matches its demand

Together, the students have the teacher's parameter count. The demand of a noise level is its sensitivity weighted by the parameter cost of the channels involved, and each phase receives a share of the budget in proportion to the demand it covers; inside a phase, the budget is then split across layers. This is one call to `scripts/dry_run_capacity_allocation.py`.

In [ ]:
#@title Size one student per phase
run("dry_run_capacity_allocation.py", "--results-json", PROFILE, "--timestep-grouping", WORK / "phases.json",
    "--allocation-results-dir", WORK / "budgets")
plan = json.loads((WORK / "budgets" / "combined_blockwise.json").read_text())["target_budget_plan"]
total = plan["total_student_system_budget"]
share = np.asarray(plan["block_budgets"]) / total
demand = compute_delta_p_eff_geomean_metric(profile)
phase_of_level = np.concatenate([[k] * (b - a) for k, (a, b) in enumerate(phases)])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.1), gridspec_kw=dict(width_ratios=[1.4, 1]))
ax1.bar(range(len(demand)), demand / demand.max(), color=[COLORS[k] for k in phase_of_level])
ax1.set_xticks([0, len(demand) - 1], ["high noise", "low noise"]); ax1.set_yticks([])
ax1.set_title("Demand at each noise level")
for row, split in enumerate([share, np.full(len(share), 1 / len(share))]):
    left = 0.0
    for k, x in enumerate(split):
        ax2.barh(row, x, left=left, color=COLORS[k], alpha=1.0 if row == 0 else 0.3, edgecolor="white")
        ax2.text(left + x / 2, row, f"{x:.0%}", ha="center", va="center", color="white" if row == 0 else "#333")
        left += x
ax2.set_yticks([0, 1], ["PACE", "equal budgets"]); ax2.invert_yaxis(); ax2.set_xticks([]); ax2.set_xlim(0, 1)
ax2.set_title(f"Split of the teacher's {total / 1e6:.1f}M parameters")
plt.tight_layout(); plt.show()

## 4. One student per denoising call

While sampling, a router sends each denoising call to the student of its phase: the nearest profiled noise level decides the phase, and only that student runs. The animation follows the sampler from pure noise to the final image.

<small>For the FFHQ teacher, the image interpolates between the frames of Figure 1 at steps 0, 10, 35 and 40.</small>

In [ ]:
#@title Watch the router while an image is sampled (about 10 seconds)
from PIL import Image as PILImage

n = T["steps"]   # EDM sampler: noise levels from 80 down to 0.002 (rho = 7)
sigma = (80 ** (1 / 7) + np.arange(n) / (n - 1) * (0.002 ** (1 / 7) - 80 ** (1 / 7))) ** 7
schedule = np.asarray(profile["sigma_values"], dtype=float)
# the router's rule (BlockwiseEDMStudent in pace/edm_distillation.py): the nearest profiled noise level decides the bin
nearest = np.abs(np.log(sigma)[:, None] - np.log(schedule)[None, :]).argmin(1)
step_bin = nearest * len(profile["sigma_bin_labels"]) // len(schedule)
step_phase = np.searchsorted(bounds, step_bin, side="right") - 1

frames = None
if T["stem"].startswith("ffhq"):   # the FFHQ trajectory of Figure 1: noise, two estimates, final sample
    figdir = REPO / "docs" / "assets" / "figures"
    frames = {s: np.asarray(PILImage.open(figdir / f"fig1-frame-{s}.png").convert("RGB"), dtype=float) / 255
              for s in (0, 10, 35, 40)}


def picture(state):
    keys = sorted(frames)
    for a, b in zip(keys, keys[1:]):
        if a <= state <= b:
            w = (state - a) / (b - a)
            return ((1 - w) * frames[a] + w * frames[b]).clip(0, 1)


gif = []
for t in list(range(n)) + [n] * 8:
    step = min(t, n - 1)
    fig = plt.figure(figsize=(9, 2.9), dpi=80)
    left = 0.04
    if frames is not None:
        ax = fig.add_axes([0.02, 0.06, 0.24, 0.8]); ax.axis("off")
        ax.imshow(picture(t * 40 / n)); ax.set_title("final sample" if t == n else "initial noise" if t == 0 else f"step {t}", fontsize=10)
        left = 0.3
    ax = fig.add_axes([left, 0.04, 0.97 - left, 0.9]); ax.axis("off"); ax.set_xlim(0, n); ax.set_ylim(0, 10)
    for s in range(n):
        ax.add_patch(plt.Rectangle((s + 0.12, 6.4), 0.76, 1.5, color=COLORS[step_phase[s]], alpha=1.0 if s <= step else 0.22))
    ax.plot([step + 0.5], [8.4], marker="v", color="#222", ms=8)
    ax.text(0, 9.3, "sampling done" if t == n else f"call {step + 1} of {n}: phase {step_phase[step] + 1}", fontsize=11)
    ax.text(0, 5.3, "high noise", fontsize=8, color="#777"); ax.text(n, 5.3, "low noise", fontsize=8, color="#777", ha="right")
    w = n / len(share)   # one card per student; the bar inside shows its share of the parameters
    for k, s in enumerate(share):
        on = t < n and k == step_phase[step]
        x0, ink = k * w, ("white" if on else "#333")
        ax.add_patch(plt.Rectangle((x0 + 0.2, 0.4), w - 0.4, 3.9, color=COLORS[k], alpha=0.95 if on else 0.15))
        ax.add_patch(plt.Rectangle((x0 + 0.8, 1.0), w - 1.6, 0.5, color=ink, alpha=0.25))
        ax.add_patch(plt.Rectangle((x0 + 0.8, 1.0), (w - 1.6) * s, 0.5, color=ink if on else COLORS[k]))
        ax.text(x0 + w / 2, 3.25, f"student {k + 1}" + (" is running" if on else ""), ha="center", fontsize=10,
                color=ink, weight="bold" if on else "normal")
        ax.text(x0 + w / 2, 2.05, f"{s:.0%} of the parameters", ha="center", fontsize=9, color=ink)
    buf = io.BytesIO(); fig.savefig(buf, format="png"); plt.close(fig)
    gif.append(PILImage.open(buf).convert("P", palette=PILImage.ADAPTIVE))

out = io.BytesIO()
gif[0].save(out, format="GIF", save_all=True, append_images=gif[1:], duration=[140] * n + [260] * 8, loop=0)
display(HTML(f'<img src="data:image/gif;base64,{base64.b64encode(out.getvalue()).decode()}" style="max-width:100%">'))
runs = [(k, int((step_phase == k).sum())) for k in range(len(share))]
print("  ".join(f"student {k + 1}: {c} of {n} calls with {share[k]:.0%} of the parameters" for k, c in runs))

## 5. Run PACE on your own teacher

The same steps, from a pretrained EDM teacher to student architectures, from a shell (step 1 needs a GPU):

```bash
git clone https://github.com/GabrieleSerussi/pace-diffusion && cd pace-diffusion && pip install -e ".[all]"
git clone https://github.com/NVlabs/edm ../edm

python scripts/evaluate_parameters_edm.py --dataset cifar10 --download --model_family vp \
    --network_pkl https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-cond-vp.pkl \
    --model-cache-dir checkpoints --trust-local-pickle --output_dir outputs/profile              # 1. profile
python scripts/optimize_timestep_grouping.py --matrix outputs/profile/results.json \
    --output outputs/grouping.json                                                            # 2. phases
python scripts/dry_run_capacity_allocation.py --results-json outputs/profile/results.json \
    --timestep-grouping outputs/grouping.json --allocation-results-dir outputs/allocations    # 3. budgets
python scripts/prepare_edm_distillation.py --results-json outputs/profile/results.json \
    --allocation-results-dir outputs/allocations --output-dir outputs/plans                   # 4. students
```

Then train the students with `scripts/train_edm_distillation.py` and sample with the router with `scripts/sample_edm_distilled.py`; `--help` lists the options of every script. Profiling every channel of a teacher takes GPU hours, which the released profiles let this notebook skip. To see the phases, budgets and routing of your own teacher here, upload its `results.json`, set `my_profile` in the setup cell and run the notebook again.

## Cite

```bibtex
@misc{kosman2026pace,
      title={Different Noise Levels, Different Needs: Discovering Diffusion Phases for Efficient Distillation},
      author={Eitan Kosman and Gabriele Serussi and Chaim Baskin},
      year={2026},
      note={Preprint},
}
```